# Pre tratamiento de datos

Cuenta de Github: https://github.com/IsacNM/manejo_datos

In [1]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/IsacNM/manejo_datos/refs/heads/main/pipol_datos.csv"
df_raw = pd.read_csv(URL)

print(f"Filas: {len(df_raw)}  |  Columnas: {list(df_raw.columns)}")
print("\nPrimeras filas:")
print(df_raw.head())
print("\nNulos por columna:")
print(df_raw.isnull().sum())


Filas: 100000  |  Columnas: ['Unnamed: 0', 'Edad', 'Género', 'Ingresos', 'Altura', 'Ciudad', 'Nivel_Educación', 'Hijos']

Primeras filas:
   Unnamed: 0  Edad Género  Ingresos  Altura    Ciudad Nivel_Educación  Hijos
0           0    82      F     62297    1.96   Phoenix             NaN      0
1           1    15      F     38674    1.83  New York             PhD      4
2           2   166    NaN     -1886    1.87       NaN       Bachelors     -5
3           3    95      M     29759    1.77   Chicago             PhD      4
4           4    36      M     99938    1.78   Phoenix             PhD      5

Nulos por columna:
Unnamed: 0             0
Edad                   0
Género             10000
Ingresos               0
Altura                 0
Ciudad             10000
Nivel_Educación    22437
Hijos                  0
dtype: int64


## 1. Valores Nulos

Los valores nulos son ausencias de datos en un conjunto de observaciones. En pandas se representan como `NaN`. Su tratamiento inadecuado puede introducir sesgos significativos en los modelos y análisis estadísticos.

### Mecanismos de Datos Faltantes

| Mecanismo | Descripción | Consecuencia |
|---|---|---|
| **MCAR** *(Missing Completely At Random)* | La ausencia no depende de ningún dato | Eliminación sin sesgo, pero reduce tamaño muestral |
| **MAR** *(Missing At Random)* | La ausencia depende de variables observadas, no del valor faltante | Imputación múltiple produce estimaciones insesgadas |
| **MNAR** *(Missing Not At Random)* | La ausencia depende del propio valor no observado | Cualquier análisis que lo ignore introducirá sesgo |

> *En el dataset `pipol_datos.csv`, las columnas `Género`, `Ciudad` y `Nivel_Educación` presentan nulos. Dado que la ausencia probablemente depende de otras variables observadas (edad, ingresos), el mecanismo más plausible es **MAR**.*

> **Implementación:** se utiliza **imputación por moda** para las variables categóricas (Género, Ciudad, Nivel_Educación), estrategia adecuada cuando la variable es nominal y el porcentaje de nulos es bajo.

In [2]:
df = df_raw.copy()

print("Nulos antes del tratamiento:")
print(df[["Género", "Ciudad", "Nivel_Educación"]].isnull().sum())

# Categórico → moda
for col in ["Género", "Ciudad", "Nivel_Educación"]:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nNulos después del tratamiento:")
print(df[["Género", "Ciudad", "Nivel_Educación"]].isnull().sum())
print("\nEjemplo de filas que tenían nulos en Género:")
print(df_raw[df_raw["Género"].isnull()].head())


Nulos antes del tratamiento:
Género             10000
Ciudad             10000
Nivel_Educación    22437
dtype: int64

Nulos después del tratamiento:
Género             0
Ciudad             0
Nivel_Educación    0
dtype: int64

Ejemplo de filas que tenían nulos en Género:
   Unnamed: 0  Edad Género  Ingresos  Altura Ciudad Nivel_Educación  Hijos
2           2   166    NaN     -1886    1.87    NaN       Bachelors     -5
5           5    -2    NaN     -1172    1.64    NaN          mastre     -5
7           7    -8    NaN    195472    1.96    NaN       Bachelors     -4
8           8    -5    NaN     -1788    1.98    NaN       Bachelors     -1
9           9   193    NaN    188965    1.52    NaN       Bachelors     -4


## 2. Valores Atípicos (Outliers)

Un **outlier** es una observación que se desvía tanto de las demás que hace sospechar que fue generada por un mecanismo diferente.

Pueden originarse por errores de medición, variabilidad natural extrema del fenómeno o registros fraudulentos. Distorsionan métricas como la media y afectan directamente el rendimiento de modelos de ML.

### Tipos de Outliers

| Tipo | Descripción | Ejemplo en `pipol_datos.csv` |
|---|---|---|
| **Puntual** | Un solo valor anómalo respecto al conjunto global | Una `Edad` de 150 años |
| **Contextual** | Anómalo en un contexto específico, normal globalmente | `Ingresos` extremos en una edad muy joven |
| **Univariado** | Anomalía en una sola variable | `Ingresos` de $10,000,000 en la encuesta |
| **Multivariado** | Anomalía en la combinación de varias variables | `Edad` de 18 años con `Ingresos` de $500,000 |

### Métodos de Detección

#### Estadísticos clásicos

- **Rango Intercuartílico (IQR)** *(Tukey, 1977)*: outlier si cae fuera de `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`. Robusto para distribuciones asimétricas. **→ Método usado en este notebook.**
- **Z-score**: estandariza la distancia a la media; umbral usual |z| > 3. Sensible a la normalidad.
- **Prueba de Grubbs**: test formal para un único outlier en distribuciones normales.

#### Basados en modelos de ML

- **Isolation Forest** *(Liu et al., 2008)*: construye árboles aleatorios; los puntos aislados más rápido tienen mayor probabilidad de ser outliers.
- **LOF (Local Outlier Factor)** *(Breunig et al., 2000)*: compara la densidad local de un punto con la de sus vecinos.
- **One-Class SVM**: aprende la frontera de la distribución normal; lo exterior es anomalía.

### Estrategias de Tratamiento

| Estrategia | Descripción | Cuándo usarla |
|---|---|---|
| **Eliminación** | Remover las observaciones atípicas | Errores de medición confirmados |
| **Winsorización** | Reemplazar por el percentil límite (P5/P95) | Reducir impacto sin eliminar datos |
| **Transformación logarítmica** | Comprime la escala en distribuciones sesgadas | Ingresos, precios, tamaños |
| **Imputación robusta** | Reemplazar por mediana o media recortada | Error sospechado pero no confirmado |

> **Implementación:** se aplica el método **IQR** sobre `Edad` e `Ingresos`, eliminando los registros fuera del rango válido. Estrategia apropiada cuando los valores extremos son probables errores de captura.

In [3]:
df = df_raw.copy()

for col in ["Edad", "Ingresos"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR

    outliers = df.loc[~df[col].between(lim_inf, lim_sup), col]
    print(f"\n--- {col} ---")
    print(f"Rango válido: [{lim_inf:.1f}, {lim_sup:.1f}]")
    print(f"Outliers encontrados: {len(outliers)}")
    print(f"Ejemplos: {outliers.head(5).tolist()}")

# Eliminar outliers en ambas columnas simultáneamente
mask = pd.Series([True] * len(df))
for col in ["Edad", "Ingresos"]:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    mask &= df[col].between(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

df_limpio = df[mask]
print(f"\nRegistros originales: {len(df)}  →  después de limpiar outliers: {len(df_limpio)}")



--- Edad ---
Rango válido: [-59.5, 160.5]
Outliers encontrados: 3930
Ejemplos: [166, 193, 188, 199, 191]

--- Ingresos ---
Rango válido: [-29164.0, 149030.0]
Outliers encontrados: 4987
Ejemplos: [195472, 188965, 161634, 158404, 171768]

Registros originales: 100000  →  después de limpiar outliers: 93040


## 3. Datos Categóricos

Las variables categóricas representan atributos cualitativos con un número finito de valores discretos. Su codificación adecuada es determinante para el rendimiento de cualquier modelo, ya que la mayoría de algoritmos requieren entradas numéricas.

### Tipos de Variables Categóricas

| Tipo | Descripción | Ejemplo en `pipol_datos.csv` |
|---|---|---|
| **Nominal** | Sin orden intrínseco entre categorías | `Género` (M/F), `Ciudad` |
| **Ordinal** | Orden natural; distancia entre categorías desconocida | `Nivel_Educación` (None < Bachelor < Master < PhD) |
| **Binaria** | Exactamente dos categorías | `Género` (M/F) |
| **Cíclica** | Estructura circular (el último regresa al primero) | Día de la semana, mes del año |

### Técnicas de Codificación

| Técnica | Descripción | Cuándo usarla |
|---|---|---|
| **One-Hot Encoding** | Variable binaria por cada categoría | Nominales con cardinalidad baja (< 15) |
| **Ordinal Encoding** | Enteros respetando el orden natural | Ordinales con orden conocido |
| **Label Encoding** | Entero único por categoría (sin orden) | Solo para árboles de decisión |
| **Target Encoding** | Media del target por categoría | Alta cardinalidad; cuidado con *data leakage* |
| **Frequency Encoding** | Reemplaza por frecuencia relativa | Alta cardinalidad; modelos lineales |
| **Embeddings neuronales** | Representaciones densas aprendidas | Redes neuronales; alta cardinalidad |
| **Codificación seno/coseno** | Proyecta al círculo unitario | Variables cíclicas (horas, días, meses) |

### Alta Cardinalidad

Cuando una variable tiene cientos de categorías únicas (ej. código postal), **One-Hot Encoding** produce matrices extremadamente dispersas. Estrategias recomendadas:
- Agrupar categorías infrecuentes en una categoría `"Otros"`.
- *Hashing trick*: proyectar a dimensión fija con función hash.
- *Target encoding* con validación cruzada para evitar *data leakage*.
- *Embeddings* aprendidos en redes neuronales *(Guo & Berkhahn, 2016)*.

> **Implementación:**
> - `Género` (nominal) → **One-Hot Encoding** con `pd.get_dummies()`.
> - `Nivel_Educación` (ordinal: None < Bachelor < Master < PhD) → **Ordinal Encoding** con mapeo explícito `{None: 0, Bachelor: 1, Master: 2, PhD: 3}`.

In [4]:
df = df_raw.copy()

# Rellenar nulos antes de codificar
for col in ["Género", "Nivel_Educación"]:
    df[col] = df[col].fillna(df[col].mode()[0])

# Nominal → One-Hot Encoding
df = pd.get_dummies(df, columns=["Género"], prefix="Genero")

# Ordinal → Encoding con jerarquía
orden = {"None": 0, "Bachelor": 1, "Master": 2, "PhD": 3}
df["Nivel_Enc"] = df["Nivel_Educación"].map(orden)

print("Columnas resultantes:")
print([c for c in df.columns])
print("\nDistribución Nivel_Enc:")
print(df["Nivel_Enc"].value_counts().sort_index())
print("\nPrimeras filas:")
print(df[["Nivel_Educación", "Nivel_Enc", "Genero_F", "Genero_M"]].head(8))


Columnas resultantes:
['Unnamed: 0', 'Edad', 'Ingresos', 'Altura', 'Ciudad', 'Nivel_Educación', 'Hijos', 'Genero_F', 'Genero_M', 'Nivel_Enc']

Distribución Nivel_Enc:
Nivel_Enc
1.0    22452
2.0    22523
3.0    45025
Name: count, dtype: int64

Primeras filas:
  Nivel_Educación  Nivel_Enc  Genero_F  Genero_M
0             PhD        3.0      True     False
1             PhD        3.0      True     False
2       Bachelors        NaN     False      True
3             PhD        3.0     False      True
4             PhD        3.0     False      True
5          mastre        NaN     False      True
6          Master        2.0      True     False
7       Bachelors        NaN     False      True


## Resumen Comparativo

| Concepto | Problema central | Herramienta de diagnóstico | Método recomendado |
|---|---|---|---|
| **Outliers** | Distorsión de estadísticas y modelos | Boxplot, Z-score, IQR | Isolation Forest, análisis contextual |
| **Valores nulos** | Sesgo y pérdida de información | Mapa de calor de nulos | MICE / Imputación múltiple |
| **Datos categóricos** | Incompatibilidad con algoritmos numéricos | Análisis de cardinalidad | One-Hot (baja), Target/Embeddings (alta) |